# Bead segmentation in SEM micrographs — training and evaluation

Runs the full pipeline for the MSc thesis *Deep Learning for Instance Segmentation and
Quantification of Bead Defects in SEM Images of Electrospun Nanofibre Mats*.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

Each stage writes its output to disk, so if Colab disconnects you can re-run only the
stage that was interrupted. Run the cells in order.

| Stage | Roughly |
|---|---|
| Setup and data preparation | 2 min |
| Classical baseline (no GPU needed) | 5 min |
| U-Net, 4 folds | 25 min |
| Mask R-CNN, 4 folds | 40 min |
| Mask R-CNN naive-split control | 40 min |
| Evaluation and tables | 3 min |

## 1 · Check the GPU

In [1]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout
      or "NO GPU - set Runtime > Change runtime type > T4 GPU, then rerun this cell.")

Tesla T4, 15360 MiB



## 2 · Mount Drive and locate the data

Upload `bead_data.zip` to the top level of your Google Drive first (My Drive).

Mounting Drive also means the results survive a disconnect: everything is written to
`MyDrive/thesis_work`, so a re-run picks up where it stopped.

In [2]:
import os, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/thesis_work")
except Exception as e:
    print(f"Drive not mounted ({e}); results will be lost on disconnect.")
    ROOT = Path("/content/thesis_work")

ROOT.mkdir(parents=True, exist_ok=True)
DATA = Path("/content/data")

zips = list(Path("/content/drive/MyDrive").glob("bead_data.zip")) if Path("/content/drive").exists() else []
if not zips:
    from google.colab import files
    print("bead_data.zip not found in My Drive — upload it now.")
    up = files.upload()
    zips = [Path(next(iter(up)))]

with zipfile.ZipFile(zips[0]) as z:
    z.extractall(DATA)
print("data:", sorted(p.name for p in (DATA / "Segmentations").iterdir()))
print("images:", len(list((DATA / "Segmentations" / "Images").glob("*.jpg"))))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data: ['Images', 'Labels']
images: 11


## 3 · Get the code

In [3]:
import os, subprocess
from pathlib import Path

# Re-derived here rather than inherited from section 2, so that this cell still
# works as the single thing to re-run after a runtime restart. A restart clears
# the kernel namespace but not /content, which is exactly the state in which the
# names below would otherwise be missing while the files they point at are fine.
DATA = Path("/content/data")
DRIVE = Path("/content/drive/MyDrive")
ROOT = DRIVE / "thesis_work" if DRIVE.exists() else Path("/content/thesis_work")

if not (DATA / "Segmentations").exists():
    raise SystemExit("run section 2 first: the micrographs are not unpacked")
if not DRIVE.exists():
    print("WARNING: Drive is not mounted. Everything this session produces will "
          "be lost on disconnect. Run section 2.")

CODE = Path("/content/thesis")
if CODE.exists():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ErlindSkura/Erlind_Skura_Thesis.git", str(CODE)], check=True)

os.environ["BEAD_DATA"] = str(DATA / "Segmentations")
os.environ["BEAD_WORK"] = str(ROOT / "work")
os.environ["BEAD_RESULTS"] = str(ROOT / "results")
os.chdir(CODE / "code")
print("cwd:", Path.cwd())
print("work:", os.environ["BEAD_WORK"])
# The pull above cannot use check=True: a shallow clone fails it for reasons that
# do not matter here. So the commit is printed instead, because the failure mode
# that costs a GPU session is a pull that quietly did nothing.
print("code:", subprocess.run(["git", "-C", str(CODE), "log", "--oneline", "-1"],
                              capture_output=True, text=True).stdout.strip())

cwd: /content/thesis/code
work: /content/drive/MyDrive/thesis_work/work


## 4 · Verify the environment

In [4]:
import importlib, subprocess, sys

# Import name -> pip name, which differ for two of these.
NEEDED = {"torch": "torch", "torchvision": "torchvision",
          "pycocotools": "pycocotools", "skimage": "scikit-image",
          "scipy": "scipy"}

for mod_name, pip_name in NEEDED.items():
    try:
        mod = importlib.import_module(mod_name)
        print(f"{mod_name:14} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"{mod_name:14} missing - installing {pip_name}")
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", pip_name],
                       check=True)

import torch
print("\ncuda available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu only")

torch          2.11.0+cu128
torchvision    0.26.0+cu128
pycocotools    ok
skimage        0.25.2
scipy          1.16.3

cuda available: True | Tesla T4


## 5 · Prepare the data

Crops the instrument banner, converts the LabelMe polygons to COCO, and writes the
fold manifests. Expect **11 images and 606 beads** — if you see anything else, stop
and check the upload.

In [5]:
!python prepare_data.py
!python folds.py

images      : 11
annotations : 606
labels      : ['bead']
written to  : /content/drive/MyDrive/thesis_work/work/coco_gt.json
images in   : /content/drive/MyDrive/thesis_work/work/images

loso:
  Z2   test=['Z2-1', 'Z2-2', 'Z2-3']
  Z4   test=['Z4-2', 'Z4-3']
  Z5   test=['Z5-1', 'Z5-2', 'Z5-3']
  Z6   test=['Z6-1', 'Z6-2', 'Z6-4']

random:
  R1   test=['Z2-2', 'Z6-1', 'Z6-2']  <-- specimen(s) ['Z2', 'Z6'] on both sides
  R2   test=['Z2-3', 'Z5-1']  <-- specimen(s) ['Z2', 'Z5'] on both sides
  R3   test=['Z4-2', 'Z4-3', 'Z5-3']  <-- specimen(s) ['Z5'] on both sides
  R4   test=['Z2-1', 'Z5-2', 'Z6-4']  <-- specimen(s) ['Z2', 'Z5', 'Z6'] on both sides


### 5b · Check the geometry

Every conversion the pipeline depends on, checked against the real annotations:
box extraction, the YOLO label export, and the dataset invariants the thesis
quotes. These are the errors that would produce plausible numbers rather than a
crash, so they have to be checked rather than assumed. All should pass.

In [6]:
!python tests.py

box geometry
  [PASS] identical boxes give IoU 1
  [PASS] half-overlapping boxes give IoU 1/3  -- 0.333333
  [PASS] disjoint boxes give IoU 0
  [PASS] empty input gives an empty matrix, not an error
  [PASS] F1 against no prediction is 0
  [PASS] F1 of a perfect prediction is 1
box extraction from masks
  [PASS] bounding box matches the mask extent  -- got [np.float64(4.0), np.float64(3.0), np.float64(11.0), np.float64(9.0)]
YOLO label export
  [PASS] every polygon becomes exactly one label line
  [PASS] vertex coordinates survive the round-trip to sub-micropixel  -- worst 5.12e-08 px on Z5-3
YOLO detection labels
  [PASS] box centre and extent match the polygon they came from  -- worst 4.90e-11
annotation invariants
  [PASS] 606 annotated particles across 11 micrographs  -- got 606
  [PASS] no vertex falls below the banner crop  -- 606/606
  [PASS] 82.8% of particles are COCO-small, as Chapter 2 states  -- 82.8% below 32x32 px
  [PASS] 1.2% are COCO-large  -- 1.2%
preprocessing
  [PAS

## 6 · Smoke test

Eight training iterations per fold. The numbers this produces are meaningless — the
point is to find out in two minutes, rather than after two hours, whether anything
crashes.

In [7]:
!python run_all.py --smoke

SMOKE RUN. Eight iterations per fold; the numbers are meaningless.
Writing to /content/drive/MyDrive/thesis_work/work_smoke
        and /content/drive/MyDrive/thesis_work/results_smoke
Any real run's predictions are left untouched.
1/6  preparing data
     11 images, 606 beads
2/6  building fold manifests
     loso: ['Z2', 'Z4', 'Z5', 'Z6']
     random: ['R1', 'R2', 'R3', 'R4']
3/6  classical baseline (leave-one-specimen-out)
[fold Z2] tuned on training partition: radius=1 min_area=10.0um2 min_dist=10 (train AJI 0.062, 6s)
    Z2-1: 1202 predicted / 71 annotated
    Z2-2: 655 predicted / 51 annotated
    Z2-3: 101 predicted / 10 annotated
[fold Z4] tuned on training partition: radius=1 min_area=10.0um2 min_dist=10 (train AJI 0.061, 4s)
    Z4-2: 674 predicted / 59 annotated
    Z4-3: 99 predicted / 4 annotated
[fold Z5] tuned on training partition: radius=1 min_area=10.0um2 min_dist=10 (train AJI 0.063, 5s)
    Z5-1: 1251 predicted / 102 annotated
    Z5-2: 762 predicted / 57 annotated

## 7 · The real run

From here the stages are separate cells. Each saves its predictions to Drive, so a
disconnect costs you one stage and not the whole run.

### 7a · Classical baseline (Otsu + watershed)

In [8]:
!python classical.py --protocol loso

[fold Z2] tuned on training partition: radius=2 min_area=10.0um2 min_dist=20 (train AJI 0.077, 81s)
    Z2-1: 349 predicted / 71 annotated
    Z2-2: 311 predicted / 51 annotated
    Z2-3: 108 predicted / 10 annotated
[fold Z4] tuned on training partition: radius=2 min_area=10.0um2 min_dist=20 (train AJI 0.079, 93s)
    Z4-2: 300 predicted / 59 annotated
    Z4-3: 103 predicted / 4 annotated
[fold Z5] tuned on training partition: radius=2 min_area=30.0um2 min_dist=20 (train AJI 0.080, 83s)
    Z5-1: 328 predicted / 102 annotated
    Z5-2: 168 predicted / 57 annotated
    Z5-3: 13 predicted / 14 annotated
[fold Z6] tuned on training partition: radius=2 min_area=30.0um2 min_dist=20 (train AJI 0.058, 81s)
    Z6-1: 329 predicted / 152 annotated
    Z6-2: 179 predicted / 62 annotated
    Z6-4: 17 predicted / 24 annotated

wrote 2205 detections to /content/drive/MyDrive/thesis_work/work/preds/loso/classical.json


### 7b · U-Net semantic baseline

In [9]:
!python train_unet.py --protocol loso --iters 1500 --batch 8

device: cuda  protocol: loso  iters: 1500  batch: 8

[fold Z2] test=['Z2-1', 'Z2-2', 'Z2-3']
    step     1/1500  loss 1.6449  3s
    step   100/1500  loss 1.3208  61s
    step   200/1500  loss 1.2485  122s
    step   300/1500  loss 1.2066  182s
    step   400/1500  loss 1.1765  242s
    step   500/1500  loss 1.1226  300s
    step   600/1500  loss 0.9405  357s
    step   700/1500  loss 1.0457  416s
    step   800/1500  loss 0.9045  475s
    step   900/1500  loss 0.8899  533s
    step  1000/1500  loss 0.8970  592s
    step  1100/1500  loss 0.9768  650s
    step  1200/1500  loss 0.9148  712s
    step  1300/1500  loss 0.9345  770s
    step  1400/1500  loss 0.8241  828s
    step  1500/1500  loss 0.9255  893s
  [Z2] 1500 steps x batch 8 = 12000 crops = 521.74 epochs
      0.502 s/step (median), 0.594 mean, 1.334 p95, first step 3.0 s
      2.88 steps/epoch, 1.7 s/epoch, 13.48 crops/s, total 893 s
    chosen on training partition: prob>=0.80, min_area=64px
    Z2-1: 44 predicted / 71 annotat

### 7c · Mask R-CNN, leave-one-specimen-out

In [10]:
!python train_maskrcnn.py --protocol loso --iters 1500 --batch 4

device: cuda  protocol: loso  iters: 1500  batch: 4

[fold Z2] train=8 test=['Z2-1', 'Z2-2', 'Z2-3']
    step     1/1500  loss 2.8568  lr 0.00003  3s
    step   100/1500  loss 0.9885  lr 0.00333  47s
    step   200/1500  loss 0.5955  lr 0.00498  89s
    step   300/1500  loss 0.8339  lr 0.00485  132s
    step   400/1500  loss 0.9887  lr 0.00459  174s
    step   500/1500  loss 0.5782  lr 0.00422  217s
    step   600/1500  loss 0.7957  lr 0.00375  261s
    step   700/1500  loss 0.5468  lr 0.00322  304s
    step   800/1500  loss 0.5842  lr 0.00265  347s
    step   900/1500  loss 0.5512  lr 0.00207  390s
    step  1000/1500  loss 0.7079  lr 0.00151  434s
    step  1100/1500  loss 0.4056  lr 0.00101  476s
    step  1200/1500  loss 0.9484  lr 0.00058  519s
    step  1300/1500  loss 0.7816  lr 0.00027  561s
    step  1400/1500  loss 0.5022  lr 0.00007  604s
    step  1500/1500  loss 0.6975  lr 0.00000  646s
  [Z2] 1500 steps x batch 4 = 6000 crops = 260.87 epochs
      0.412 s/step (median), 0

### 7d · Faster R-CNN, detection only

The same backbone, anchors and schedule as 7c, without the mask branch. The
laboratory endpoint is a count, and counting needs only detection, so this
measures what the mask branch is actually worth for the quantity of interest.
It reports box AP rather than mask AP, and no AJI — it predicts no masks.

In [11]:
!python train_fasterrcnn.py --protocol loso --iters 1500 --batch 4

device: cuda  protocol: loso  iters: 1500  batch: 4

[fold Z2] train=8 test=['Z2-1', 'Z2-2', 'Z2-3']
    step     1/1500  loss 1.6590  lr 0.00003  5s
    step   100/1500  loss 0.5411  lr 0.00333  42s
    step   200/1500  loss 0.2559  lr 0.00498  81s
    step   300/1500  loss 0.4743  lr 0.00485  118s
    step   400/1500  loss 0.6616  lr 0.00459  156s
    step   500/1500  loss 0.2891  lr 0.00422  193s
    step   600/1500  loss 0.4794  lr 0.00375  232s
    step   700/1500  loss 0.2033  lr 0.00322  268s
    step   800/1500  loss 0.2539  lr 0.00265  307s
    step   900/1500  loss 0.2433  lr 0.00207  344s
    step  1000/1500  loss 0.4103  lr 0.00151  382s
    step  1100/1500  loss 0.0971  lr 0.00101  417s
    step  1200/1500  loss 0.6149  lr 0.00058  455s
    step  1300/1500  loss 0.4210  lr 0.00027  492s
    step  1400/1500  loss 0.1816  lr 0.00007  529s
    step  1500/1500  loss 0.4117  lr 0.00000  566s
  [Z2] 1500 steps x batch 4 = 6000 crops = 260.87 epochs
      0.353 s/step (median), 0

### 7e · YOLOv8

A different family of architecture, not a controlled ablation: YOLOv8 shares no
backbone, head, loss or augmentation with the R-CNN models, so it answers "does
another family do better here", not "which component is responsible".

Three settings differ from the Ultralytics defaults, each for a measured reason.
`imgsz=1024` keeps native resolution — the default 640 would shrink the median
500× particle from 14.3 px to 8.9 px. `mosaic=0.0` is off because mosaic roughly
halves apparent object size, and 82.8% of these particles are already below
COCO's small-object threshold. `max_det=400` prevents truncating dense images.

`-seg` weights predict masks and join the mask comparison; swap to `yolov8s.pt`
for detection only.

In [12]:
!pip install -q ultralytics
!python train_yolo.py --protocol loso --iters 1500 --batch 4 --weights yolov8s-seg.pt

Streaming output truncated to the last 5000 lines.

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
    555/667      7.07G      1.438      2.572     0.9646      1.065          0         64       1024: 100% ━━━━━━━━━━━━ 3/3 3.8it/s 0.8s

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
    556/667      7.07G      1.268      2.398     0.8374      1.017          0         53       1024: 100% ━━━━━━━━━━━━ 3/3 3.6it/s 0.8s

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
    557/667      7.07G      1.477      2.498       1.02      1.098          0         90       1024: 100% ━━━━━━━━━━━━ 3/3 3.2it/s 0.9s

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
    558/667      7.07G      1.299      2.341     0.8924      1.022          0         63       1024: 100% ━━━━━━━━━━━━ 3/3 3.3it/s 0.9s



### 7f · YOLOv5

Two things to be clear about when you present this.

It is **YOLOv5u**, not the YOLOv5 of the 2020 papers: Ultralytics ships the
YOLOv5 backbone fitted with YOLOv8's anchor-free head. So `yolov5su` against
`yolov8s` varies mostly the *backbone* — it is not the anchor-based versus
anchor-free comparison the version numbers suggest.

Ultralytics has no `-seg` variant for YOLOv5, so it detects only: box AP and
counting, no AJI and no size distribution. It writes to its own prediction file,
so it does not overwrite 7e.

In [ ]:
!python train_yolo.py --protocol loso --iters 1500 --batch 4 --weights yolov5su.pt

Streaming output truncated to the last 5000 lines.
      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    284/750      2.54G      1.924      1.428      1.299        131       1024: 100% ━━━━━━━━━━━━ 2/2 5.6it/s 0.4s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    285/750      2.54G      2.049      1.476      1.279        292       1024: 100% ━━━━━━━━━━━━ 2/2 5.1it/s 0.4s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    286/750      2.54G      1.857      1.406      1.221        181       1024: 100% ━━━━━━━━━━━━ 2/2 5.5it/s 0.4s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    287/750      2.54G      1.904      1.363      1.264        209       1024: 100% ━━━━━━━━━━━━ 2/2 4.9it/s 0.4s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    288/750      2.54G      2.121      1.576      1.333        220       1024: 100% ━━━━━━━━━━━━ 2/

### 7g · Preprocessing ablation

The same architecture, schedule, folds and threshold rule — only the input
transform changes, so a difference is attributable to preprocessing alone.

Contrast measured on the annotations *before* training — separation between
particle and mat (Δ), divided by background spread (σ), which is what a filter
actually has to work with:

| Variant | Δ (grey levels) | σ | Δ/σ | vs baseline |
|---|---|---|---|---|
| none | 18.0 | 33.2 | 0.550 | — |
| median | 18.3 | 30.5 | 0.612 | +11% |
| background | 20.3 | 34.9 | 0.593 | +8% |
| clahe | 23.1 | 62.5 | 0.372 | −32% |

CLAHE is the case to understand: it gives the **widest** raw gap of the four,
23.1 levels against 18.0, and on that number alone would be the obvious choice.
It also nearly doubles the background spread, because it amplifies the fibre
texture as much as the particles. Usable contrast falls by a third.

This measures contrast, not accuracy. A CNN is not a threshold — it can use
texture and shape the statistic ignores — so treat it as a prior on where to
look, not a prediction. Each variant is a full 4-fold run, about an hour apiece.

In [6]:
!python train_maskrcnn.py --protocol loso --iters 1500 --batch 4 --preprocess median
!python train_maskrcnn.py --protocol loso --iters 1500 --batch 4 --preprocess clahe
!python train_maskrcnn.py --protocol loso --iters 1500 --batch 4 --preprocess background

device: cuda  protocol: loso  iters: 1500  batch: 4

[fold Z2] train=8 test=['Z2-1', 'Z2-2', 'Z2-3']
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth
100% 177M/177M [00:00<00:00, 204MB/s]
    step     1/1500  loss 2.8878  lr 0.00003  7s
    step   100/1500  loss 0.9622  lr 0.00333  49s
    step   200/1500  loss 0.6023  lr 0.00498  90s
    step   300/1500  loss 0.8196  lr 0.00485  132s
    step   400/1500  loss 0.9969  lr 0.00459  173s
    step   500/1500  loss 0.5814  lr 0.00422  215s
    step   600/1500  loss 0.7999  lr 0.00375  257s
    step   700/1500  loss 0.5587  lr 0.00322  298s
    step   800/1500  loss 0.6209  lr 0.00265  341s
    step   900/1500  loss 0.5609  lr 0.00207  382s
    step  1000/1500  loss 0.7416  lr 0.00151  424s
    step  1100/1500  loss 0.4423  lr 0.00101  465s
    step  1200/1500  loss 0.9704  lr 0.00058  506s
    step  1300/1500  loss 0

### 7h · Mask R-CNN under the naive random split

This is the control for the data-leakage contribution: the same model and the same
fold sizes, but the split ignores specimen identity. The gap between this and 7c is
the amount a naive protocol would have overstated performance.

In [1]:
!python train_maskrcnn.py --protocol random --iters 1500 --batch 4

python3: can't open file '/content/train_maskrcnn.py': [Errno 2] No such file or directory


## 8 · Evaluate and build the Chapter 5 tables

In [ ]:
!python evaluate.py
!python make_tables.py

## 9 · Look at the results

In [ ]:
import json, os
from pathlib import Path

RES = Path(os.environ["BEAD_RESULTS"])
m = json.loads((RES / "metrics.json").read_text())

g = m["ground_truth"]
print(f"ground truth: {g['n']} beads, median diameter {g['median']:.2f} um, "
      f"IQR [{g['iqr'][0]:.2f}, {g['iqr'][1]:.2f}]")

for protocol, methods in m.items():
    if protocol == "ground_truth":
        continue
    print(f"\n=== {protocol} ===")
    for name, r in methods.items():
        o = r["overall"]

        # Same convention as evaluate.py: a metric the method cannot support
        # prints as '--'. Faster R-CNN and YOLOv5 predict boxes and no masks, so
        # evaluate.py leaves AJI, PQ and the merge/split rates unset rather than
        # zero, and the key is absent from `overall` entirely. Printing 0.000
        # there would be a fabricated number in the one place this notebook is
        # read from.
        def _m(key, fmt=".3f", width=0, o=o):
            return (format(o[key]["mean"], fmt) if key in o else "--").rjust(width)

        sd = f"+-{o['ap50']['std']:.3f}" if "ap50" in o else " " * 9
        print(f"{name:11s} [{r['iou_type']:4s}] AP50 {_m('ap50')}{sd}   "
              f"AP {_m('ap')}   AJI {_m('aji', width=5)}   "
              f"PQ {_m('pq', width=5)}   "
              f"|count err| {_m('abs_counting_error', '.1f', 5)}%   "
              f"merge {_m('merge_rate', '.1f', 4)}%   "
              f"split {_m('split_rate', '.1f', 4)}%")

print("\n[segm] is mask AP, [bbox] is box AP. The two columns are not "
      "comparable:\nbox AP is the easier measure, so a box-only method's AP "
      "must not be read\nas beating a mask method's.")

In [ ]:
import os
from pathlib import Path

print((Path(os.environ["BEAD_RESULTS"]) / "chapter5_tables.tex").read_text())

## 10 · Download everything

`thesis_results.zip` contains `metrics.json`, the generated LaTeX tables, the
figures, and — importantly — the raw per-image predictions.

The predictions are included because every metric in the thesis is a *function*
of them. Shipping them back means a new metric (a different IoU threshold, a
size-stratified AP, a counting statistic nobody has thought of yet) can be
computed on a laptop in seconds instead of costing another GPU session. They
cost well under a megabyte per method. An earlier version of this notebook left
them behind on the Colab VM, and when the session was reset the numbers could
only be recovered by retraining.

In [ ]:
import os, shutil
from pathlib import Path
from google.colab import files

RES = Path(os.environ["BEAD_RESULTS"])
out = Path("/content/thesis_results")
shutil.rmtree(out, ignore_errors=True)
out.mkdir()
shutil.copytree(RES, out / "results", dirs_exist_ok=True)
figs = RES.parent / "figures"
if figs.exists():
    shutil.copytree(figs, out / "figures", dirs_exist_ok=True)

# The predictions and the ground-truth COCO file: together these make every
# metric recomputable offline, with no GPU and no retraining.
work = Path(os.environ["BEAD_WORK"])
if (work / "preds").exists():
    shutil.copytree(work / "preds", out / "preds", dirs_exist_ok=True)
for extra in ("coco_gt.json", "folds_loso.json", "folds_random.json"):
    if (work / extra).exists():
        shutil.copy(work / extra, out / extra)

shutil.make_archive("/content/thesis_results", "zip", out)
print("bundled:", sorted(p.name for p in out.rglob("*") if p.is_file())[:20])
files.download("/content/thesis_results.zip")